In [1]:
from pymatgen.core.structure import Structure

# import dask

import time
import shutil
import yaml
import subprocess
import os.path, os
from pathlib import Path

import ase
import ase.io
import ase.io.espresso
from ase.data import atomic_masses, atomic_numbers
import json

import pandas as pd

from util_qe import get_k_point_density, get_q_point_density, \
        update_dict_relax, \
        update_dict_ph_elph, \
        update_q2r_matdyn_elph, \
        write_qe_file, \
        get_k_q_grid


In [2]:
par_el = 'run_full'
root = '/blue/hennig/jasongibson/diff_model'
root = f'{root}/materials/{par_el}/mp_relaxed/'
df = pd.read_pickle(f'pkl_files/df_cpd_{par_el}_strict.pkl')
df = df.loc[df.tcad_cpd > 5]
len(df)

1363

In [3]:
def submit_qe_cal(jobdir):
    k_point_density = get_k_point_density() #min kpoints per inv A
    q_point_density = get_q_point_density() #min qpoints per inv A

    if not Path(jobdir+"/a2F.dos20").is_file():
        if not Path(jobdir+"/relax.out").is_file():
            print(jobdir)
            filename = jobdir+"/CONTCAR"

            atom_obj = ase.io.read(filename)

            cell = atom_obj.cell
            reciprocal_cell = cell.reciprocal()
            k_grid, q_grid = get_k_q_grid(k_point_density,q_point_density,reciprocal_cell)
            # devnull.write("k_grid "+str(k_grid)+" q_grid "+str(q_grid)+"\n")


            with open("default_qe_para.json","r") as fp:
                para = json.load(fp)
            relax_parameter = para["relax_parameter"]

            relax_parameter = update_dict_relax(relax_parameter,k_grid,atom_obj)
            write_qe_file(param=relax_parameter,filename=jobdir+"/relax.in")


        # print()
        filename = jobdir+"/relax.out"
        atom_obj = ase.io.read(filename)

        cell = atom_obj.cell
        reciprocal_cell = cell.reciprocal()
        k_grid, q_grid = get_k_q_grid(k_point_density,q_point_density,reciprocal_cell)


        with open("default_qe_para.json","r") as fp:
            para = json.load(fp)

        relax_parameter = para["relax_parameter"]
        relax_parameter = update_dict_relax(relax_parameter,k_grid,atom_obj)

        scf_dense = relax_parameter.copy()
        scf_dense["CONTROL"]["calculation"] = "scf"
        scf_dense["SYSTEM"]["la2F"] = True
        scf_dense["ELECTRONS"]["conv_thr"] = 1.0e-12
        scf_coarse = scf_dense.copy()

        ph_elph = para["ph"]
        q2r = para["q2r"]
        matdyn = para["matdyn_dos"]

        write_qe_file(param=scf_dense,filename=jobdir+"/scf_dense.in")

        ph_elph = update_dict_ph_elph(ph_elph, q_grid,atom_obj)
        q2r, matdyn = update_q2r_matdyn_elph(q2r, matdyn, q_grid,atom_obj)

        ph_elph["INPUTPH"]["nmix_ph"] = 8
        ph_elph["INPUTPH"]["diagonalization"] = "cg"
        ph_elph["INPUTPH"]["recover"] = True
        
        # ph_elph["INPUTPH"]["niter_ph"] = 400
        
        
        write_qe_file(param=ph_elph,filename=jobdir+"/ph_elph.in")
        write_qe_file(param=q2r,filename=jobdir+"/q2r.in")
        write_qe_file(param=matdyn,filename=jobdir+"/matdyn.in")
        return True


In [4]:
# submit_qe_cal('/blue/hennig/jasongibson/elemental_sub/materials/B/mp_relaxed/53642')

In [5]:
from tqdm.notebook import tqdm
is_new = []
for line in tqdm(df.index):
    if str(line) != '5378':
        i = submit_qe_cal(root+str(line))
        if i:
            is_new.append(line)
    else:
        print(line)

  0%|          | 0/1363 [00:00<?, ?it/s]

5378


In [6]:
len(is_new)

681

In [7]:
df = df.loc[is_new]

In [8]:
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.io.ase  import AseAtomsAdaptor as aaa
def get_sgn(df):
    analyzer = SpacegroupAnalyzer(aaa.get_structure(df.structure))
    sym_data = analyzer.get_symmetry_dataset()
    return sym_data['number']

In [9]:
tqdm.pandas()

In [10]:
df['sgn'] = df.progress_apply(get_sgn,axis=1)

  0%|          | 0/681 [00:00<?, ?it/s]

/blue/hennig/jasongibson/envs/e3nn/lib/python3.9/site-packages/spglib/spglib.py:115: DeprecationWarning: dict interface (SpglibDataset['number']) is deprecated.Use attribute interface ({self.__class__.__name__}.{key}) instead
  warnings.warn(


In [58]:
df = df.sort_values(['sgn','natoms','tcad_cpd'],ascending=[False,True,False])#[['sgn','natoms','tcad_cpd']].head(30)

In [60]:
with open(root+'list_ele_ph', "w") as file:
    for item in df.index:
        # Write each item to the file followed by a newline
        file.write(f"{item}\n")

In [11]:
df_filt.sort_values('tcad_cpd',ascending=False)[[ 'formula','eah','tcad_cpd','tcad_m3g','sgn']].head(20)

,formula,eah,tcad_cpd,tcad_m3g,sgn
5378,GaNb6Zn,0.053626,19.163834,10.960787,200
8631,Nb3Ti,0.054936,18.973925,16.368372,123
11416,IrTc2,-0.000783,16.849359,5.283441,139
206,NbTiV4,0.036531,16.839470,15.916989,216
121451,Ga2Nb5V,0.088274,16.450375,7.153118,6
172024,CrTc6Ti,0.022930,16.380442,9.455545,200
64238,Mo5NbOs2,-0.020084,16.328726,13.197006,115
63350,Nb10Zr2,0.043820,16.224724,15.004932,2
12496,Nb5Zr,0.055346,16.170770,13.888534,38
149313,Mo5NbOsRu,-0.010022,16.076816,12.723314,25


In [9]:
import os
import glob

In [10]:
root = '/blue/hennig/jasongibson/elemental_sub/materials/alex/mp_relaxed/'
dirs = glob.glob(root + '**/phonon_2x2x2/temp', recursive=True)

In [11]:
os.path.dirname(os.path.dirname(dirs[0]))

'/blue/hennig/jasongibson/elemental_sub/materials/alex/mp_relaxed/2120066'

In [12]:
dirs[0]

'/blue/hennig/jasongibson/elemental_sub/materials/alex/mp_relaxed/2120066/phonon_2x2x2/temp'

In [45]:
target_dir = os.path.dirname(dirs[0])
target_file = os.path.join(target_dir, 'phonon_2x2x2.in')
with open(target_file, 'r') as file:
    lines = file.readlines()

In [46]:
lines

['&INPUTPH\n',
 "\tprefix = 'Co2Nb2Ti8',\n",
 "\tfildyn = 'Co2Nb2Ti8.dyn',\n",
 '\treduce_io = .true.,\n',
 "\toutdir = 'temp/',\n",
 '\tldisp = .true.,\n',
 '\ttrans = .true.,\n',
 "\tfildvscf = 'dv',\n",
 '\tnq1 = 2,\n',
 '\tnq2 = 2,\n',
 '\tnq3 = 2,\n',
 '\ttr2_ph = 1e-16,\n',
 '\tnmix_ph = 20,\n',
 '/\n']

In [41]:
lines.insert(-1, '\trecover=.true.\n')

In [47]:
if lines[-2] != '\trecover=.true.\n':
    print('yes')
else:
    print('no')

yes


In [42]:
lines

['&INPUTPH\n',
 "\tprefix = 'Co2Nb2Ti8',\n",
 "\tfildyn = 'Co2Nb2Ti8.dyn',\n",
 '\treduce_io = .true.,\n',
 "\toutdir = 'temp/',\n",
 '\tldisp = .true.,\n',
 '\ttrans = .true.,\n',
 "\tfildvscf = 'dv',\n",
 '\tnq1 = 2,\n',
 '\tnq2 = 2,\n',
 '\tnq3 = 2,\n',
 '\ttr2_ph = 1e-16,\n',
 '\tnmix_ph = 20,\n',
 '\trecover=.true.\n',
 '/\n']

In [48]:
# Traverse directories to find the pattern
for dirpath in dirs:
    # Go one directory up
    target_dir = os.path.dirname(dirpath)
    target_file = os.path.join(target_dir, 'phonon_2x2x2.in')

    # Check if the target file exists
    if os.path.isfile(target_file):
        # Read the file content
        with open(target_file, 'r') as file:
            lines = file.readlines()

        # Add 'recover=.true.' to the second-to-last line
        if lines[-2] != '\trecover=.true.\n':
            lines.insert(-1, '\trecover=.true.\n')

            # Write back the modified content to the file
            with open(target_file, 'w') as file:
                file.writelines(lines)

        print(f"Modified: {target_file}")
    else:
        print(f"File not found: {target_file}")

Modified: /blue/hennig/jasongibson/elemental_sub/materials/alex/mp_relaxed/2320905/phonon_2x2x2/phonon_2x2x2.in
Modified: /blue/hennig/jasongibson/elemental_sub/materials/alex/mp_relaxed/2325762/phonon_2x2x2/phonon_2x2x2.in
Modified: /blue/hennig/jasongibson/elemental_sub/materials/alex/mp_relaxed/534398/phonon_2x2x2/phonon_2x2x2.in
Modified: /blue/hennig/jasongibson/elemental_sub/materials/alex/mp_relaxed/169526/phonon_2x2x2/phonon_2x2x2.in
Modified: /blue/hennig/jasongibson/elemental_sub/materials/alex/mp_relaxed/701726/phonon_2x2x2/phonon_2x2x2.in
Modified: /blue/hennig/jasongibson/elemental_sub/materials/alex/mp_relaxed/214295/phonon_2x2x2/phonon_2x2x2.in
Modified: /blue/hennig/jasongibson/elemental_sub/materials/alex/mp_relaxed/2320166/phonon_2x2x2/phonon_2x2x2.in
Modified: /blue/hennig/jasongibson/elemental_sub/materials/alex/mp_relaxed/412692/phonon_2x2x2/phonon_2x2x2.in
Modified: /blue/hennig/jasongibson/elemental_sub/materials/alex/mp_relaxed/681343/phonon_2x2x2/phonon_2x2x2.i

In [52]:
inds = []
for dirpath in dirs:
    inds.append(dirpath.split('/')[-3])

In [54]:
root

'/blue/hennig/jasongibson/elemental_sub/materials/alex/mp_relaxed/'

In [56]:
with open(root+'list_ele', "w") as file:
    for item in inds:
        # Write each item to the file followed by a newline
        file.write(f"{item}/phonon_2x2x2\n")